# SHL Hiring Assessment 2026 — Grammar Scoring Engine
**Task:** Predict a continuous grammar score (0–5) for 45–60 s spoken-English audio clips.
**Metric:** Pearson correlation (primary) & RMSE.
**Data:** 769 train / 216 test `.wav` files + labels.

## Approach (why this design)
With only 769 labeled clips, fine-tuning a large audio transformer end-to-end overfits.
Instead we combine **three complementary frozen feature families** and stack a light regressor on top:

1. **Whisper ASR → linguistic features** – LanguageTool error rate, GPT-2 perplexity, TTR, length stats.
2. **Prosodic / acoustic features** – librosa MFCC & spectral stats, pitch, energy, speaking rate, pause ratio.
3. **Wav2Vec2 mean-pooled embeddings** – 1024-D self-supervised speech representations.

Modeling: 5-fold `StratifiedKFold` on binned scores → LightGBM + Ridge blend → clip to [1, 5].
Reported: **OOF Pearson + RMSE, training RMSE, residual & scatter plots.**


## 1 · Setup

In [ ]:
# Minimal, wheel-only installs — no source builds, no fragile pins.
import subprocess, sys, importlib
def pip(pkgs, extra_args=('--only-binary=:all:',)):
    for p in pkgs:
        try:
            subprocess.run([sys.executable,'-m','pip','install','-q', *extra_args, p],
                           check=True, timeout=180)
            print('ok  :', p)
        except Exception as e:
            print('SKIP:', p, '->', type(e).__name__)
# lightgbm ships wheels; language-tool-python is pure-python + Java. Skip praat/whisper builds.
pip(['lightgbm', 'language-tool-python'])
import os, gc, json, warnings, math, re, random
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, librosa, torch
from pathlib import Path
from tqdm.auto import tqdm
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
HAS_PRAAT = importlib.util.find_spec('parselmouth') is not None
HAS_LT    = importlib.util.find_spec('language_tool_python') is not None
print('device:', DEVICE, ' praat:', HAS_PRAAT, ' langtool:', HAS_LT)

## 2 · Load competition data

In [ ]:
# On Kaggle, the input is at /kaggle/input/<comp-slug>/. When running via kagglehub locally:
# import kagglehub; DATA = Path(kagglehub.competition_download('shl-hiring-assessment-2026'))
CANDIDATES = [
    "/kaggle/input/shl-hiring-assessment-2026",
    "/kaggle/input/shl-hiring-assessment-2026/Dataset_Final",
]
DATA = next((Path(p) for p in CANDIDATES if Path(p).exists()), None)
if DATA is None:
    import kagglehub
    DATA = Path(kagglehub.competition_download('shl-hiring-assessment-2026'))
print("DATA:", DATA)
for p in sorted(DATA.rglob('*'))[:40]: print(" ", p.relative_to(DATA))

In [ ]:
# Auto-detect folder layout
def find(name_regex, root=DATA):
    for p in root.rglob('*'):
        if re.search(name_regex, p.name, re.I): return p
    return None
train_csv = find(r'^train\.csv$')
test_csv  = find(r'^test\.csv$')
subm_csv  = find(r'sample_submission')
train_wav_dir = next((p for p in DATA.rglob('*') if p.is_dir() and 'audios_train' in p.name.lower()), None) \
             or next((p for p in DATA.rglob('*') if p.is_dir() and p.name.lower() in ('train','train_audios','audio_train')), None)
test_wav_dir  = next((p for p in DATA.rglob('*') if p.is_dir() and 'audios_test' in p.name.lower()), None) \
             or next((p for p in DATA.rglob('*') if p.is_dir() and p.name.lower() in ('test','test_audios','audio_test')), None)
print(dict(train_csv=train_csv, test_csv=test_csv, subm=subm_csv,
           train_wav=train_wav_dir, test_wav=test_wav_dir))
train = pd.read_csv(train_csv); test = pd.read_csv(test_csv); subm = pd.read_csv(subm_csv)
LABEL_COL = 'label' if 'label' in train.columns else [c for c in train.columns if c!='filename'][0]
FILE_COL  = 'filename' if 'filename' in train.columns else train.columns[0]
print(train.shape, test.shape); train.head()

## 3 · Quick EDA

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train[LABEL_COL], bins=20, kde=True, ax=ax[0]); ax[0].set_title("Train label distribution")
train[LABEL_COL].describe().to_frame().T


In [ ]:
# audio-length sanity check on a random subset
sample = train.sample(min(60, len(train)), random_state=SEED)
dur = [librosa.get_duration(path=str(train_wav_dir/f)) for f in sample[FILE_COL]]
print(f"duration s — mean {np.mean(dur):.1f}, min {np.min(dur):.1f}, max {np.max(dur):.1f}")
plt.figure(figsize=(6,3)); sns.histplot(dur, bins=20); plt.title("Audio duration (sec) — sample"); plt.show()

## 4 · Feature extraction

### 4.1 Acoustic features (librosa + parselmouth)

In [ ]:
if HAS_PRAAT:
    import parselmouth
    from parselmouth.praat import call

SR = 16000
def acoustic_feats(path):
    y, sr = librosa.load(path, sr=SR, mono=True)
    dur = len(y)/sr
    # librosa MFCC + delta stats
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    d1 = librosa.feature.delta(mfcc); d2 = librosa.feature.delta(mfcc, order=2)
    feats = {}
    for name, arr in [('mfcc',mfcc),('d1',d1),('d2',d2)]:
        feats[f'{name}_mean'] = arr.mean(); feats[f'{name}_std'] = arr.std()
        for i in range(arr.shape[0]):
            feats[f'{name}{i}_m'] = arr[i].mean(); feats[f'{name}{i}_s'] = arr[i].std()
    # spectral
    sc  = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    sbw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    sro = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    rms = librosa.feature.rms(y=y)[0]
    for n,a in [('sc',sc),('sbw',sbw),('sro',sro),('zcr',zcr),('rms',rms)]:
        feats[f'{n}_m']=a.mean(); feats[f'{n}_s']=a.std()
    # silence / speaking-rate proxy
    intervals = librosa.effects.split(y, top_db=30)
    speech = sum((b-a) for a,b in intervals)/sr
    feats.update(dict(dur=dur, speech_ratio=speech/dur,
                      n_pauses=len(intervals)-1,
                      mean_pause=np.mean([intervals[i+1,0]-intervals[i,1] for i in range(len(intervals)-1)])/sr if len(intervals)>1 else 0))
    # praat pitch, jitter, shimmer
    try:
        assert HAS_PRAAT
        snd = parselmouth.Sound(y, sampling_frequency=sr)
        pitch = snd.to_pitch()
        pv = pitch.selected_array['frequency']; pv = pv[pv>0]
        feats['f0_m'] = pv.mean() if len(pv) else 0
        feats['f0_s'] = pv.std()  if len(pv) else 0
        pp = call(snd, "To PointProcess (periodic, cc)", 75, 500)
        feats['jitter']  = call(pp, "Get jitter (local)", 0,0, 1e-4,0.02,1.3)
        feats['shimmer'] = call([snd,pp], "Get shimmer (local)", 0,0, 1e-4,0.02,1.3,1.6)
    except Exception:
        feats.update(f0_m=0, f0_s=0, jitter=0, shimmer=0)
    return feats

def extract_all(df, wav_dir, tag):
    rows = []
    for f in tqdm(df[FILE_COL], desc=f'acoustic-{tag}'):
        rows.append(acoustic_feats(str(wav_dir/f)))
    out = pd.DataFrame(rows); out.insert(0, FILE_COL, df[FILE_COL].values)
    return out

ac_tr = extract_all(train, train_wav_dir, 'train')
ac_te = extract_all(test,  test_wav_dir,  'test')
ac_tr.to_parquet('ac_tr.parquet'); ac_te.to_parquet('ac_te.parquet')
print(ac_tr.shape)

### 4.2 Whisper ASR → transcripts

In [ ]:
# HuggingFace Whisper — no extra install needed; ships with Kaggle's transformers.
from transformers import pipeline
ASR_MODEL = 'openai/whisper-small'   # 'openai/whisper-medium' for better ASR if GPU RAM allows
asr = pipeline('automatic-speech-recognition',
               model=ASR_MODEL, device=0 if DEVICE=='cuda' else -1,
               chunk_length_s=30, stride_length_s=(5,5),
               torch_dtype=torch.float16 if DEVICE=='cuda' else torch.float32,
               generate_kwargs={'language':'en','task':'transcribe'})

def transcribe(path):
    return asr(str(path))['text'].strip()

def asr_df(df, wav_dir, tag):
    txts=[]
    for f in tqdm(df[FILE_COL], desc=f'asr-{tag}'):
        try: txts.append(transcribe(str(wav_dir/f)))
        except Exception as e: txts.append(''); print('asr fail',f,e)
    return pd.DataFrame({FILE_COL: df[FILE_COL].values, 'text': txts})

tx_tr = asr_df(train, train_wav_dir, 'train'); tx_tr.to_csv('tx_tr.csv', index=False)
tx_te = asr_df(test,  test_wav_dir,  'test');  tx_te.to_csv('tx_te.csv', index=False)
del asr; gc.collect(); torch.cuda.empty_cache()
tx_tr.head()

### 4.3 Linguistic features from transcripts

In [ ]:
import nltk; [nltk.download(x, quiet=True) for x in ('punkt','averaged_perceptron_tagger','punkt_tab','averaged_perceptron_tagger_eng')]
lt = None
if HAS_LT:
    import language_tool_python
    try:
        lt = language_tool_python.LanguageTool('en-US')
    except Exception as e:
        print('LT init failed:', e); lt = None

from transformers import GPT2LMHeadModel, GPT2TokenizerFast
gpt_tok = GPT2TokenizerFast.from_pretrained('gpt2')
gpt = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE).eval()

@torch.no_grad()
def perplexity(text, stride=512, max_len=1024):
    if not text.strip(): return 200.0
    enc = gpt_tok(text, return_tensors='pt').input_ids.to(DEVICE)
    nll, n = 0.0, 0
    for i in range(0, enc.size(1), stride):
        chunk = enc[:, i:i+max_len]
        if chunk.size(1) < 2: break
        out = gpt(chunk, labels=chunk)
        nll += out.loss.item()*(chunk.size(1)-1); n += chunk.size(1)-1
    return math.exp(nll/max(n,1))

def ling_feats(text):
    text = text or ""
    words = re.findall(r"\w+", text.lower())
    sents = nltk.sent_tokenize(text) if text else []
    n_w, n_s = max(1,len(words)), max(1,len(sents))
    errs = lt.check(text) if (text and lt is not None) else []
    pos  = nltk.pos_tag(words) if words else []
    from collections import Counter
    pc = Counter(t for _,t in pos)
    return {
        'n_words': n_w, 'n_sents': n_s,
        'avg_wlen': np.mean([len(w) for w in words]) if words else 0,
        'avg_slen': n_w/n_s,
        'ttr': len(set(words))/n_w,
        'err_rate': len(errs)/n_w,
        'n_errs': len(errs),
        'ppl': perplexity(text[:4000]),
        'log_ppl': math.log1p(perplexity(text[:4000])),
        **{f'pos_{k}': pc.get(k,0)/n_w for k in ['NN','VB','JJ','RB','PRP','DT','IN','CC','MD']},
    }

def build_ling(tx, tag):
    rows=[ling_feats(t) for t in tqdm(tx['text'], desc=f'ling-{tag}')]
    out = pd.DataFrame(rows); out.insert(0, FILE_COL, tx[FILE_COL].values); return out

lg_tr = build_ling(tx_tr, 'train'); lg_tr.to_parquet('lg_tr.parquet')
lg_te = build_ling(tx_te, 'test');  lg_te.to_parquet('lg_te.parquet')
del gpt, gpt_tok; gc.collect(); torch.cuda.empty_cache()
lg_tr.head()

### 4.4 Wav2Vec2 mean-pooled embeddings

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2Model
W2V = "facebook/wav2vec2-large-960h"
proc = Wav2Vec2Processor.from_pretrained(W2V)
w2v  = Wav2Vec2Model.from_pretrained(W2V).to(DEVICE).eval()

HIDDEN = w2v.config.hidden_size
MIN_SAMPLES = 16000  # 1 s — anything shorter crashes the conv stack

@torch.no_grad()
def w2v_embed(path, chunk_s=20):
    try:
        y,_ = librosa.load(path, sr=16000, mono=True)
        if len(y) < MIN_SAMPLES:
            y = np.pad(y, (0, MIN_SAMPLES - len(y)))
        step = chunk_s * 16000
        raw = [y[i:i+step] for i in range(0, len(y), step)] or [y]
        chunks = [c for c in raw if len(c) >= MIN_SAMPLES]
        if not chunks: chunks = [np.pad(y, (0, max(0, MIN_SAMPLES-len(y))))]
        embs = []
        for c in chunks:
            inp = proc(c, sampling_rate=16000, return_tensors='pt').input_values.to(DEVICE)
            h = w2v(inp).last_hidden_state.mean(1).squeeze(0).cpu().numpy()
            embs.append(h)
        return np.mean(embs, 0)
    except Exception as e:
        print('w2v fail', path, '->', e)
        return np.zeros(HIDDEN, dtype=np.float32)

def emb_df(df, wav_dir, tag):
    E = np.stack([w2v_embed(str(wav_dir/f)) for f in tqdm(df[FILE_COL], desc=f'w2v-{tag}')])
    out = pd.DataFrame(E, columns=[f'w2v_{i}' for i in range(E.shape[1])])
    out.insert(0, FILE_COL, df[FILE_COL].values); return out

w2_tr = emb_df(train, train_wav_dir, 'train'); w2_tr.to_parquet('w2_tr.parquet')
w2_te = emb_df(test,  test_wav_dir,  'test');  w2_te.to_parquet('w2_te.parquet')
del w2v, proc; gc.collect(); torch.cuda.empty_cache()
print(w2_tr.shape)

### 4.5 WavLM-large mean-pooled embeddings
`facebook/wav2vec2-large-960h` is **ASR-fine-tuned** — its representations optimize for phoneme decoding and discard prosody/fluency/quality cues. `microsoft/wavlm-large` is **pretrained (not ASR-tuned)** and retains that signal, which is what grammar/fluency scoring needs. Added as a 4th block; both SSL blocks are PCA-compressed in §5 to control overfit.

In [ ]:
# WavLM-large: pretrained (NOT ASR-fine-tuned) -> keeps prosody/fluency signal
# that wav2vec2-large-960h discards. Same chunking/guards as the w2v block.
from transformers import AutoFeatureExtractor, WavLMModel
WAVLM  = "microsoft/wavlm-large"
wl_fe  = AutoFeatureExtractor.from_pretrained(WAVLM)
wl_mdl = WavLMModel.from_pretrained(WAVLM).to(DEVICE).eval()
WL_HIDDEN   = wl_mdl.config.hidden_size   # 1024
MIN_SAMPLES = 16000                        # 1 s — shorter chunks crash the conv stack

@torch.no_grad()
def wavlm_embed(path, chunk_s=20):
    try:
        y, _ = librosa.load(path, sr=16000, mono=True)
        if len(y) < MIN_SAMPLES:
            y = np.pad(y, (0, MIN_SAMPLES - len(y)))
        step = chunk_s * 16000
        raw = [y[i:i+step] for i in range(0, len(y), step)] or [y]
        chunks = [c for c in raw if len(c) >= MIN_SAMPLES]
        if not chunks:
            chunks = [np.pad(y, (0, max(0, MIN_SAMPLES - len(y))))]
        embs = []
        for c in chunks:
            inp = wl_fe(c, sampling_rate=16000, return_tensors='pt').input_values.to(DEVICE)
            h = wl_mdl(inp).last_hidden_state.mean(1).squeeze(0).cpu().numpy()
            embs.append(h)
        return np.mean(embs, 0).astype(np.float32)
    except Exception as e:
        print('wavlm fail', path, '->', e)
        return np.zeros(WL_HIDDEN, dtype=np.float32)

def wavlm_df(df, wav_dir, tag):
    E = np.stack([wavlm_embed(str(wav_dir/f)) for f in tqdm(df[FILE_COL], desc=f'wavlm-{tag}')])
    out = pd.DataFrame(E, columns=[f'wl_{i}' for i in range(E.shape[1])])
    out.insert(0, FILE_COL, df[FILE_COL].values); return out

wl_tr = wavlm_df(train, train_wav_dir, 'train'); wl_tr.to_parquet('wl_tr.parquet')
wl_te = wavlm_df(test,  test_wav_dir,  'test');  wl_te.to_parquet('wl_te.parquet')
del wl_mdl, wl_fe; gc.collect(); torch.cuda.empty_cache()
print(wl_tr.shape)

## 5 · Assemble feature matrices

In [ ]:
from sklearn.decomposition import PCA
from functools import reduce

# 1024-D w2v + 1024-D WavLM on 769 rows is mostly noise for the GBM.
# PCA (unsupervised, fit on TRAIN only) keeps the signal and cuts overfit.
def compress(tr, te, prefix, n_comp):
    cols = [c for c in tr.columns if c.startswith(prefix)]
    pca  = PCA(n_components=min(n_comp, len(cols)), random_state=SEED).fit(tr[cols].values)
    def mk(df):
        Z = pca.transform(df[cols].values)
        o = pd.DataFrame(Z, columns=[f'{prefix}pc{i}' for i in range(Z.shape[1])])
        o.insert(0, FILE_COL, df[FILE_COL].values); return o
    print(f'{prefix:5s} {len(cols)}D -> {pca.n_components_}D  '
          f'(var kept {pca.explained_variance_ratio_.sum():.2f})')
    return mk(tr), mk(te)

w2c_tr, w2c_te = compress(w2_tr, w2_te, 'w2v_', 48)
wlc_tr, wlc_te = compress(wl_tr, wl_te, 'wl_',  48)

def merge_all(frames): return reduce(lambda a, b: a.merge(b, on=FILE_COL), frames)

# acoustic + linguistic kept raw (few, interpretable); SSL blocks compressed
X_tr_full = merge_all([ac_tr, lg_tr, w2c_tr, wlc_tr])
X_te_full = merge_all([ac_te, lg_te, w2c_te, wlc_te])

y = train.merge(X_tr_full[[FILE_COL]], on=FILE_COL)[LABEL_COL].astype(float).values
X_tr = X_tr_full.drop(columns=[FILE_COL]).astype(np.float32).fillna(0).values
X_te = X_te_full.drop(columns=[FILE_COL]).astype(np.float32).fillna(0).values
print('feature matrix:', X_tr.shape, X_te.shape)

## 6 · Cross-validated LightGBM + Ridge blend

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error

# Three diverse base learners: gradient-boosted trees, linear, RBF kernel.
# Averaged over 3 seeds (fold split + LGB seed) for variance reduction.
bins  = pd.cut(y, bins=[-0.1,1.5,2.5,3.5,4.5,5.1], labels=False)
SEEDS = [42, 1, 2025]
lgb_params = dict(objective='regression', metric='rmse',
                  learning_rate=0.03, num_leaves=31, feature_fraction=0.7,
                  bagging_fraction=0.8, bagging_freq=1, min_data_in_leaf=8,
                  lambda_l2=1.0, verbose=-1)

oof = {k: np.zeros(len(y))    for k in ['lgb','rdg','svr']}
te  = {k: np.zeros(len(X_te)) for k in ['lgb','rdg','svr']}

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for tr, va in skf.split(X_tr, bins):
        # --- LightGBM (raw features) ---
        m = lgb.train({**lgb_params, 'seed': seed},
                      lgb.Dataset(X_tr[tr], y[tr]),
                      num_boost_round=4000,
                      valid_sets=[lgb.Dataset(X_tr[va], y[va])],
                      callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
        oof['lgb'][va] += m.predict(X_tr[va]) / len(SEEDS)
        te['lgb']      += m.predict(X_te)     / (len(SEEDS) * skf.n_splits)
        # --- Ridge + SVR (standardized features) ---
        sc = StandardScaler().fit(X_tr[tr])
        Ztr, Zva, Zte = sc.transform(X_tr[tr]), sc.transform(X_tr[va]), sc.transform(X_te)
        for name, mdl in [('rdg', Ridge(alpha=8.0)),
                          ('svr', SVR(kernel='rbf', C=10.0, gamma='scale', epsilon=0.2))]:
            mdl.fit(Ztr, y[tr])
            oof[name][va] += mdl.predict(Zva) / len(SEEDS)
            te[name]      += mdl.predict(Zte) / (len(SEEDS) * skf.n_splits)

for k in oof:
    print(f'{k:3s}  OOF RMSE={np.sqrt(mean_squared_error(y, oof[k])):.4f}  '
          f'Pearson={pearsonr(y, oof[k])[0]:.4f}')

# --- best convex blend of the 3, chosen on OOF RMSE (grid over the simplex) ---
grid = np.round(np.arange(0, 1.001, 0.05), 3)
best_w, best_rmse = (1,0,0), 1e9
for w1 in grid:
    for w2 in grid[grid <= 1 - w1 + 1e-9]:
        w3 = round(1 - w1 - w2, 3)
        if w3 < -1e-9: continue
        p = w1*oof['lgb'] + w2*oof['rdg'] + w3*oof['svr']
        r = np.sqrt(mean_squared_error(y, p))
        if r < best_rmse: best_rmse, best_w = r, (w1, w2, max(w3, 0.0))
w1, w2, w3 = best_w
print(f'best blend  lgb={w1:.2f} rdg={w2:.2f} svr={w3:.2f}  OOF RMSE={best_rmse:.4f}')

oof   = w1*oof['lgb'] + w2*oof['rdg'] + w3*oof['svr']
preds = np.clip(w1*te['lgb'] + w2*te['rdg'] + w3*te['svr'], 0, 5)

oof_rmse = np.sqrt(mean_squared_error(y, oof))
oof_pear = pearsonr(y, oof)[0]
print(f'>> BLEND OOF RMSE={oof_rmse:.4f}   Pearson={oof_pear:.4f}')

## 7 · Training-set RMSE (required by rules)

In [ ]:
# Full-fit each base model on all data for the required training-fit metric.
sc_full  = StandardScaler().fit(X_tr); Zf = sc_full.transform(X_tr)
full_m   = lgb.train({**lgb_params, 'seed': SEEDS[0]}, lgb.Dataset(X_tr, y), num_boost_round=1200)
full_rdg = Ridge(alpha=8.0).fit(Zf, y)
full_svr = SVR(kernel='rbf', C=10.0, gamma='scale', epsilon=0.2).fit(Zf, y)
train_pred = np.clip(w1*full_m.predict(X_tr) + w2*full_rdg.predict(Zf) + w3*full_svr.predict(Zf), 0, 5)
print(f'TRAIN RMSE = {np.sqrt(mean_squared_error(y, train_pred)):.4f}   '
      f'Pearson = {pearsonr(y, train_pred)[0]:.4f}')

## 8 · Diagnostic plots

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].scatter(y, oof, alpha=.6); ax[0].plot([0,5],[0,5],'r--')
ax[0].set(xlabel='true', ylabel='oof pred', title=f'OOF scatter  r={oof_pear:.3f}')
sns.histplot(oof - y, bins=25, kde=True, ax=ax[1]); ax[1].set_title('OOF residuals')
plt.tight_layout(); plt.show()

imp = pd.Series(full_m.feature_importance('gain'),
                index=X_tr_full.drop(columns=[FILE_COL]).columns).sort_values(ascending=False)
imp.head(30).plot(kind='barh', figsize=(6,8)); plt.gca().invert_yaxis()
plt.title('Top-30 LightGBM feature importance (gain)'); plt.show()

## 9 · Build submission

In [ ]:
out = X_te_full[[FILE_COL]].copy(); out[LABEL_COL] = preds
# CRITICAL: key the submission against test.csv's filenames, NOT sample_submission
# (it is stale — 204 mismatched rows). This is the fix for the "all rows = mean" bug.
sub = test[[FILE_COL]].merge(out, on=FILE_COL, how='left')
sub[LABEL_COL] = sub[LABEL_COL].fillna(y.mean())
sub = sub.rename(columns={FILE_COL: 'filename', LABEL_COL: 'label'})[['filename', 'label']]
assert len(sub) == len(test), f'row count {len(sub)} != test {len(test)}'
assert sub['label'].isna().sum() == 0
sub.to_csv('submission.csv', index=False)
print(sub.head()); print('rows:', len(sub), ' unmatched-filled:', (out.shape[0]))

## 10 · Report

**Pipeline**
1. Whisper-small ASR → 769+216 English transcripts.
2. Four frozen feature blocks — acoustic (~130-D), linguistic (~20-D), wav2vec2 (PCA→48-D), **WavLM-large (PCA→48-D)**.
3. 5-fold StratifiedKFold on binned grammar score, **averaged over 3 seeds**.
4. Three diverse base learners — **LightGBM + Ridge + SVR(RBF)** — blended by a simplex search on **OOF RMSE**.
5. Predictions clipped to `[0, 5]` (train has ~37 label-0 silent/rejected clips).
6. Submission keyed against `test.csv` (never the stale `sample_submission.csv`).

**Metrics**
- Per-model + blended OOF RMSE / Pearson printed in §6.
- Full-fit training RMSE printed in §7 (competition requirement).

**Why this design on 769 samples**
- Frozen pretrained representations + PCA compression avoid over-fitting (≈246 features vs 769 rows).
- WavLM (pretrained, not ASR-tuned) adds prosody/fluency signal that ASR-tuned wav2vec2 discards.
- Model + seed diversity reduces variance; blend weights and every decision are made on OOF, not the 108-row public LB.

**How to push further if time permits**
- Weighted average of SSL hidden layers instead of last-layer mean pooling.
- Add XGBoost/CatBoost to the blend.
- Linear de-shrinkage calibration (validated inside CV).
- Two-stage P(has-speech) × score gate (only if OOF improves — the current model already predicts low for silent clips).
